# CALENDARIO

In [1]:
import base64
import io
import mimetypes

import requests
from langchain_community.document_loaders import (
    PDFPlumberLoader,
    UnstructuredImageLoader,
)
from openai import OpenAI
from pydantic import BaseModel, Field


/tmp/ipykernel_93289/2944699992.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


In [ ]:
filepath = "/home/mattemindev/Repositories/github.com/mattemindev/bordeus/ingestion/old_knowledge/sub-ato-e/_comuni/donnas/calendario/giu_nov_2026.jpeg"

file_url = "https://teknoserviceitalia.b-cdn.net/wp-content/uploads/2026/05/BARD-DONNAS-HONE-UD.pdf"

response = requests.get(file_url)
response.raise_for_status()

pdf = io.BytesIO(response.content)


# 1. Definisci lo schema esatto che desideri ottenere
class CalendarioRifiuti(BaseModel):
    organico: list[str] = Field(
        description="Lista di date ISO (AAAA-MM-GG) per la raccolta dell'organico"
    )
    rur: list[str] = Field(
        description="Lista di date ISO (AAAA-MM-GG) per il RUR / Indifferenziato"
    )
    cc: list[str] = Field(
        description="Lista di date ISO (AAAA-MM-GG) per carta e cartoni"
    )
    vetro: list[str] = Field(description="Lista di date ISO (AAAA-MM-GG) per il vetro")
    ipm: list[str] = Field(
        description="Lista di date ISO (AAAA-MM-GG) per imballaggi in plastica e metalli"
    )


# 2. Inizializza il modello multimodale locale tramite LangChain
#ollama_model = "gemma4:latest"
ollama_model="qwen3.5:9b"
#ollama_model="glm-ocr:latest"

# Configured by environment variables
ollama_base_url = "http://localhost:11434/v1"
ollama_api_key = "ollama"

client = OpenAI(api_key=ollama_api_key, base_url=ollama_base_url)

# 3. Codifica l'immagine in Base64
image_mime_type, _ = mimetypes.guess_type(filepath)
if image_mime_type is None:
    raise ValueError("Could not determine the image MIME type")
with open(filepath, "rb") as image_file:
    image_base64 = base64.b64encode(image_file.read()).decode("utf-8")
data_uri = f"data:{image_mime_type};base64,{image_base64}"

# 4. Invia la richiesta al modello chiedendo la conversione in date ISO (considerando l'anno 2026 visibile in tabella)
system_prompt = """
"""

user_prompt = """
Analizza attentamente l'immagine del calendario di raccolta differenziata per il periodo giugno 2026 - novembre 2026.
Estrai i giorni di passaggio per ciascuna categoria di rifiuto e convertili in un elenco di date in formato ISO standard (AAAA-MM-GG).
Ad esempio, se sotto 'GIUGNO '26' vedi '1 LUN ORGANICO', la data corrispondente è '2026-06-01'.
Mappa tutte le date per ogni singola categoria presente.
"""

# Esecuzione del parsing visivo strutturato
response = client.responses.parse(
    model=ollama_model,
    input=[
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": [
                {
                    "type": "input_image",
                    "image_url": data_uri,
                }
            ],
        },
    ],
    text_format=CalendarioRifiuti,
)


# Ora hai il dizionario pronto da salvare nel tuo Vector Store o Database per il RAG!
calendario_rifiuti = response.output_parsed
print(calendario_rifiuti)




KeyboardInterrupt: 

In [37]:
import base64

from openai import OpenAI
from pydantic import BaseModel, Field


# 1. Define Pydantic schema for structured JSON output
class WasteCategoryGroup(BaseModel):
    category: str = Field(
        description="Waste category name in Italian (e.g., ORGANICO, RUR, CARTA E CARTONI, VETRO, IMB. PLAST. E METALLI)"
    )
    dates: list[str] = Field(
        description="List of collection dates for this category (e.g., ['01/06/2026', '03/06/2026'])"
    )


class WasteCalendarData(BaseModel):
    calendar_period: str = Field(
        description="Time period covered by the calendar, e.g., 'Giugno 2026 - Novembre 2026'"
    )
    grouped_schedule: list[WasteCategoryGroup] = Field(
        description='Collection days grouped by waste category (e.g., {"category":"ORGANICO","dates":["03/06/2026"]})'
    )


# Helper function to convert local image file to base64 format
def encode_image(image_path: str) -> str:
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


def extract_calendar(image_path: str) -> WasteCalendarData:
    # ollama_model = "gemma4:latest"
    #ollama_model = "qwen3.5:9b"
    ollama_model="ornith-1.5:9b"

    # Configured by environment variables
    ollama_base_url = "http://localhost:11434/v1"
    ollama_api_key = "ollama"

    client = OpenAI(api_key=ollama_api_key, base_url=ollama_base_url)
    base64_image = encode_image(image_path)

    response = client.beta.chat.completions.parse(
        model=ollama_model,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            "Extract all waste collection dates from this Italian calendar image. The calendar has been organized in a spreadsheet form."
                            "In the calendar, each month is organized as column. Each column has a Header describing the reference month and year (e.g., GIUGNO '26), and for each day of the month, the corresponding waste category (e.g., 1 LUN | ORGANICO)."
                            "Group the dates by category (e.g., ORGANICO, RUR, CARTA E CARTONI, VETRO, IMB. PLAST. E METALLI). Each category has his own color (e.g., ORGANICO orange, RUR grey, IMB. PLAST. E METALLI yellow, ...)."
                            "Ensure each date includes the full date format (DD/MM/YYYY) matching the corresponding month header (GIUGNO'26 through NOVEMBRE'26 columns). "
                            ""
                        ),
                    },
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{base64_image}"},
                    },
                ],
            }
        ],
        reasoning_effort="low",
        response_format=WasteCalendarData,
    )

    return response.choices[0].message.parsed


image_file_path = "/home/mattemindev/Repositories/github.com/mattemindev/bordeus/ingestion/old_knowledge/sub-ato-e/_comuni/donnas/calendario/giu_nov_2026.jpeg"

extracted_schedule = extract_calendar(image_file_path)

print(extracted_schedule.model_dump_json(indent=2))

KeyboardInterrupt: 

# OPENROUTER

In [ ]:
import base64

from openai import OpenAI
from pydantic import BaseModel, Field


# 1. Define Pydantic schema for structured JSON output
class WasteCategoryGroup(BaseModel):
    category: str = Field(
        description="Waste category name in Italian (e.g., ORGANICO, RUR, CARTA E CARTONI, VETRO, IMB. PLAST. E METALLI)"
    )
    dates: list[str] = Field(
        description="List of collection dates for this category (e.g., ['01/06/2026', '03/06/2026'])"
    )


class WasteCalendarData(BaseModel):
    calendar_period: str = Field(
        description="Time period covered by the calendar, e.g., 'Giugno 2026 - Novembre 2026'"
    )
    grouped_schedule: list[WasteCategoryGroup] = Field(
        description='Collection days grouped by waste category (e.g., {"category":"ORGANICO","dates":["03/06/2026"]})'
    )


# Helper function to convert local image file to base64 format
def encode_image(image_path: str) -> str:
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


def extract_calendar(image_path: str) -> WasteCalendarData:
    model = "dots-studio/dots-3-note-preview:free"

    # Configured by environment variables
    base_url = "https://openrouter.ai/api/v1"
    api_key = ""

    client = OpenAI(api_key=api_key, base_url=base_url)
    base64_image = encode_image(image_path)

    response = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            "Extract all waste collection dates from this Italian calendar image. The calendar has been organized in a spreadsheet form."
                            "In the calendar, each month is organized as column. Each column has a Header describing the reference month and year (e.g., GIUGNO '26), and for each day of the month, the corresponding waste category (e.g., 1 LUN | ORGANICO)."
                            "Group the dates by category (e.g., ORGANICO, RUR, CARTA E CARTONI, VETRO, IMB. PLAST. E METALLI). Each category has his own color (e.g., ORGANICO orange, RUR grey, IMB. PLAST. E METALLI yellow, ...)."
                            "Ensure each date includes the full date format (DD/MM/YYYY) matching the corresponding month header (GIUGNO'26 through NOVEMBRE'26 columns). "
                            "Answer in the requested format. "
                        ),
                    },
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{base64_image}"},
                    },
                ],
            }
        ],
        extra_body={"reasoning": {"enabled": True}},
        response_format=WasteCalendarData,
    )

    return response.choices[0].message.parsed


image_file_path = "/home/mattemindev/Repositories/github.com/mattemindev/bordeus/ingestion/old_knowledge/sub-ato-e/_comuni/donnas/calendario/dic_mag_2027.jpeg"

extracted_schedule = extract_calendar(image_file_path)

print(extracted_schedule.model_dump_json(indent=2))

{
  "calendar_period": "DICEMBRE '26 through MAGGIO '27 (December 2026 to May 2027)",
  "grouped_schedule": [
    {
      "category": "ORGANICO (Orange)",
      "dates": [
        "07/12/2026",
        "14/12/2026",
        "21/12/2026",
        "28/12/2026",
        "04/01/2027",
        "11/01/2027",
        "18/01/2027",
        "25/01/2027",
        "01/02/2027",
        "08/02/2027",
        "15/02/2027",
        "22/02/2027",
        "01/03/2027",
        "08/03/2027",
        "15/03/2027",
        "22/03/2027",
        "29/03/2027",
        "01/04/2027",
        "05/04/2027",
        "08/04/2027",
        "12/04/2027",
        "15/04/2027",
        "19/04/2027",
        "22/04/2027",
        "26/04/2027",
        "29/04/2027",
        "03/05/2027",
        "06/05/2027",
        "10/05/2027",
        "13/05/2027",
        "17/05/2027",
        "20/05/2027",
        "24/05/2027",
        "26/05/2027",
        "27/05/2027",
        "31/05/2027"
      ]
    },
    {
      "category"

In [50]:
markdown_content = ""
markdown_content += f"# Calendario Porta a Porta - {extracted_schedule.calendar_period}\n"
for schedule in extracted_schedule.grouped_schedule:
    markdown_content += f"## {schedule.category}\n"
    for date in schedule.dates:
        markdown_content += f"- {date}\n"
    markdown_content += "\n"

print(markdown_content)

# Calendario Porta a Porta - DICEMBRE '26 through MAGGIO '27 (December 2026 to May 2027)
## ORGANICO (Orange)
- 07/12/2026
- 14/12/2026
- 21/12/2026
- 28/12/2026
- 04/01/2027
- 11/01/2027
- 18/01/2027
- 25/01/2027
- 01/02/2027
- 08/02/2027
- 15/02/2027
- 22/02/2027
- 01/03/2027
- 08/03/2027
- 15/03/2027
- 22/03/2027
- 29/03/2027
- 01/04/2027
- 05/04/2027
- 08/04/2027
- 12/04/2027
- 15/04/2027
- 19/04/2027
- 22/04/2027
- 26/04/2027
- 29/04/2027
- 03/05/2027
- 06/05/2027
- 10/05/2027
- 13/05/2027
- 17/05/2027
- 20/05/2027
- 24/05/2027
- 26/05/2027
- 27/05/2027
- 31/05/2027

## RUR (Grey)
- 01/12/2026
- 15/12/2026
- 22/12/2026
- 29/12/2026
- 12/01/2027
- 26/01/2027
- 09/02/2027
- 23/02/2027
- 09/03/2027
- 23/03/2027
- 06/04/2027
- 20/04/2027
- 04/05/2027
- 18/05/2027

## CARTA E CARTONI (Blue)
- 11/12/2026
- 25/12/2026
- 01/01/2027
- 07/01/2027
- 22/01/2027
- 05/02/2027
- 19/02/2027
- 04/03/2027
- 19/03/2027
- 02/04/2027
- 16/04/2027
- 30/04/2027
- 14/05/2027
- 28/05/2027

## VETRO (Green

In [1]:
from paddleocr import TextDetection

model = TextDetection(model_name="PP-OCRv5_server_det")
output = model.predict(input="/home/mattemindev/Repositories/github.com/mattemindev/bordeus/ingestion/old_knowledge/sub-ato-e/_comuni/donnas/calendario/giu_nov_2026.jpeg", batch_size=1)
for res in output:
    res.print()
    res.save_to_img(save_path="./output/")
    res.save_to_json(save_path="./output/res.json")


Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/mattemindev/.paddlex/official_models/PP-OCRv5_server_det`.


NotImplementedError: (Unimplemented) ConvertPirAttribute2RuntimeAttribute not support [pir::ArrayAttribute<pir::DoubleAttribute>]  (at /paddle/paddle/fluid/framework/new_executor/instruction/onednn/onednn_instruction.cc:116)


In [26]:
filepath = "/home/mattemindev/Repositories/github.com/mattemindev/bordeus/ingestion/old_knowledge/sub-ato-e/_comuni/donnas/calendario/giu_nov_2026.jpeg"

loader = UnstructuredImageLoader(filepath,
                                 strategy="hi_res",
                                 languages=["ita"])
loaded_content = loader.load()

for doc in loaded_content:
    print(doc.page_content)

TLUN [MNIORGANICOMANAN 1 MER | iis-PLAST:EMETALH | _1S48 | o o | 1] o o o o —| 160 (soreanicogz] 100 ___ Tic Cee ea el o: AaeeNnNneZ3a Elle HEK-F.SS,IIII,6EEeEMoEII*”.”.—-—-Lo SVI 3 MER | IMB-PLAST-EMETALLI | SVEN |__| SLUN [frORGANICONEANS] 3GI0 [fORGANICO[A{N| 3 SA6 | vEmRO I] 3 man fr___RUR | 4 GIOV [ifORGANICORENARII «sa | o o o o o | ame| | 4ven| egseluoli | saoom)______________| «MER | Ims-PLAST'EMETALI SVEN| | Spoomf__________| smeel____1I sie fiero: SUN evo sso { __a"»«* _Ò; 6SA | | 6LUN [ffmORGANICONSENS] 6GO |mmorcanicoReNI soom|____y\yvsmaee | sue |__| moom |__| mei o | Tver | ecu | 7TLUN [ORGANICORAA] ? er | ims-prast EMETALUI | 7548 |__| 8 LUN |MMMMMIORGANICOMENNS Mer |__| 8SAB [vErROS SN [RU] sO e orcaNIcogeee] “00m |__ i | 860 [ffmmoReANIcORESSS] s0om|____________| SsmeR | iMs:PLASTEMETAUI | SVEN[ |] SLUN [fORcANIcO Ra ‘o Mer |__| ven | euzcueli | 10 LUN |EMORGANICONEANN| 10 cio |mmmworcaNicoReSI 10ss8 |__| ome __ | n GIOV [OR ANIGONSI tisoc [geeseseseve se coeseesei tiv

In [ ]:
filepath = "/home/mattemindev/Repositories/github.com/mattemindev/bordeus/ingestion/old_knowledge/sub-ato-e/_comuni/donnas/calendario/wp-content-uploads-2026-05-BARD-DONNAS-HONE-UD.pdf"

loader = PDFPlumberLoader(filepath, extract_images=True)
loaded_content = loader.load()

for doc in loaded_content:
    print(doc.page_content)

[INFO] 2026-08-25 15:03:41,259 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-25 15:03:41,268 [RapidOCR] download_file.py:60: File exists and is valid: /home/mattemindev/Repositories/github.com/mattemindev/bordeus/.venv/lib/python3.12/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-08-25 15:03:41,269 [RapidOCR] main.py:63: Using /home/mattemindev/Repositories/github.com/mattemindev/bordeus/.venv/lib/python3.12/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-08-25 15:03:41,337 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-25 15:03:41,339 [RapidOCR] download_file.py:60: File exists and is valid: /home/mattemindev/Repositories/github.com/mattemindev/bordeus/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-25 15:03:41,339 [RapidOCR] main.py:63: Using /home/mattemindev/Repositories/github.com/mattemindev/bordeus/.venv/lib/python3.12/site-packages/rapidocr/mo

CONTATTI
Calendario raccolta rifiuti 2026-2027
APP: SUBATO-E DIFFERENZIA
MAIL: SUBATO.E@TEKNOSERVICEITALIA.COM
BARD-DONNAS-HÔNE
NUMERO VERDE: 800079960
NUMERO WHATSAPP 347 429 6380
PER INFO, RITIRO INGOMBRANTI E LAVAGGIO CASSONETTI.
UTENZE DOMESTICHE E CONDOMINI
RICORDIAMO ALLE UTENZE DI ESPORRE IL RIFIUTO (NEI CASSONETTI, MASTELLI O SACCHI IN DOTAZIONE) COME PREVISTO DAL CALENDARIO DI RACCOLTA, DALLE ORE 19.00 DELLA SERA PRECEDENTE ALLE ORE 05.00 DEL GIORNO DELLA RACCOLTA E
PROVVEDERE AL RITIRO AD AVVENUTO PASSAGGIO.
MODALITÀ DI
CONFERIMENTO
Conferire l’organico chiuso all’interno
di sacchetti compostabili.
Conferire la carta sfusa. Compattare
le scatole e non appallottolare le
buste tipo quelle del pane.
Conferire il cartone piegato e legato,
oppure stoccato negli appositi roll
container. Sempre ridotto di volume.
Conferire, riducendo il volume, gli
imballaggi in plastica e matalli sfusi
all’interno dei cassonetti in dotazione.
Conferire il vetro sfuso e senza
coperchio metallico.
IM